In [1]:
!pip install -q transformers datasets evaluate seqeval matplotlib seaborn scikit-learn pytorch-crf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [2]:
# Load library
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from torchcrf import CRF
from IPython.display import display

In [3]:
# config
CHOSEN_MODEL = 'distibert-base-cased'
DATASET_FILE = '/content/dataset.jsonl'

RUN_MODE = 'ALL'

print(f"Model type: {CHOSEN_MODEL.upper()}")
print(f"Run mode: {RUN_MODE}")

Model type: DISTIBERT-BASE-CASED
Run mode: ALL


In [6]:
# load and prepare data
dataset = load_dataset('json', data_files=DATASET_FILE, split='train')
dataset = dataset.train_test_split(test_size=0.2, seed=42)

all_tags = [tag for tags_list in dataset["train"]["ner_tags"] for tag in tags_list]
unique_tags = sorted(list(set(all_tags)))

label2id = {label: i for i, label in enumerate(unique_tags)}
id2label = {i: label for i, label in enumerate(unique_tags)}

def encode_tags(example):
  example["ner_tags_id"] = [label2id.get(tag, 0) for tag in example["ner_tags"]]
  return example

dataset = dataset.map(encode_tags)

# cal class weight for focal loss
tag_counts = pd.Series(all_tags).value_counts()
total_tags = len(all_tags)
class_weights = [total_tags / (len(unique_tags) * tag_counts[label]) for label in unique_tags]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

In [7]:
# define losses and trainers
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss_raw = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss_raw)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss_raw
        if self.weight is not None:
            alpha = self.weight[targets]
            focal_loss = focal_loss * alpha
        return focal_loss.mean()

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        active_loss = inputs["attention_mask"].view(-1) == 1
        active_logits = logits.view(-1, model.config.num_labels)
        active_labels = torch.where(active_loss, labels.view(-1), torch.tensor(-100, device=labels.device))

        valid_indices = active_labels != -100
        valid_logits = active_logits[valid_indices]
        valid_labels = active_labels[valid_indices]

        focal_loss_fn = FocalLoss(weight=class_weights_tensor.to(model.device), gamma=2.0)
        loss = focal_loss_fn(valid_logits, valid_labels)
        return (loss, outputs) if return_outputs else loss

# DICE LOSS
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6, ignore_index=-100):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        num_classes = logits.shape[-1]
        probs = F.softmax(logits, dim=-1).view(-1, num_classes)
        targets = targets.view(-1)

        valid_mask = targets != self.ignore_index
        valid_targets = targets[valid_mask]
        valid_probs = probs[valid_mask]

        if valid_targets.numel() == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        targets_one_hot = F.one_hot(valid_targets, num_classes=num_classes).float()
        intersection = torch.sum(valid_probs * targets_one_hot, dim=0)
        cardinality = torch.sum(valid_probs + targets_one_hot, dim=0)

        dice_score = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return (1. - dice_score).mean()

class DiceLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        dice_loss_fn = DiceLoss(ignore_index=-100)
        loss = dice_loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

#  CE + DICE LOSS TRAINER
class CEDiceLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

        labels = inputs.get("labels")

        outputs = model(**inputs)

        ce_loss = outputs.loss
        logits = outputs.logits

        dice_loss_fn = DiceLoss(ignore_index=-100)
        dice_loss = dice_loss_fn(logits, labels)

        total_loss = ce_loss + dice_loss

        return (total_loss, outputs) if return_outputs else total_loss

In [9]:
# CRF architechture
class TransformerCRF(nn.Module):
    def __init__(self, model_name, num_labels, use_focal_loss=False, gamma=2.0):
        super(TransformerCRF, self).__init__()
        self.num_labels = num_labels
        self.use_focal_loss = use_focal_loss
        self.gamma = gamma
        self.config = AutoConfig.from_pretrained(model_name)
        self.transformer = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.classifier(outputs.last_hidden_state)

        loss = None
        if labels is not None:
            mask = attention_mask.bool()
            safe_labels = torch.where(labels == -100, torch.tensor(0, device=labels.device), labels)

            if self.use_focal_loss:
                # Sequence-level Focal Loss
                nll = -self.crf(logits, tags=safe_labels, mask=mask, reduction='none')
                pt = torch.exp(-nll)
                loss = (((1 - pt) ** self.gamma) * nll).mean()
            else:
                # Standard CRF NLL Loss
                loss = -self.crf(logits, tags=safe_labels, mask=mask, reduction='mean')

        eval_mask = attention_mask.bool()
        decoded_tags = self.crf.decode(logits, mask=eval_mask)

        fake_logits = torch.zeros_like(logits)
        for i, tags in enumerate(decoded_tags):
            for j, tag in enumerate(tags):
                fake_logits[i, j, tag] = 1.0

        return {"loss": loss, "logits": fake_logits} if loss is not None else {"logits": fake_logits}

In [10]:
# evaluation metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
